In [1]:
import os
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

In [2]:
decoder_dirs = {
    "Simple Tanh": "STanh_TargetAlpha",
    "Simple Sigmoid": "SSigmoid_TargetAlpha",
    "MR Tanh": "MRTanh_TargetAlpha",
    "MR Sigmoid": "MRSigmoid_TargetAlpha"
}

# Load CSVs for all decoders
decoder_data = {}
for name, path in decoder_dirs.items():
    training_csv = os.path.join(path, "training_summary.csv")
    eval_csv = os.path.join(path, "decoder_eval_summary.csv")
    
    if os.path.exists(training_csv) and os.path.exists(eval_csv):
        train_df = pd.read_csv(training_csv)
        eval_df = pd.read_csv(eval_csv)
        decoder_data[name] = {"train": train_df, "eval": eval_df}
    else:
        print(f"Warning: CSVs not found for {name}")

In [3]:
results = {}

for name, data in decoder_data.items():
    df = data["eval"].copy()
    best_seed_df = df.loc[df.groupby("alpha")["recon_acc"].idxmax()].reset_index(drop=True)
    best_seed_df["task_awareness_score"] = best_seed_df["recon_acc"] / best_seed_df["recon_CE"]

    best_score_row = best_seed_df.loc[best_seed_df["task_awareness_score"].idxmax()]
    best_acc_row = best_seed_df.loc[best_seed_df["recon_acc"].idxmax()]
    best_ce_row = best_seed_df.loc[best_seed_df["recon_CE"].idxmin()]
    best_compression_row = best_seed_df.loc[best_seed_df["compression_pct"].idxmax()]

    stats = df.groupby("alpha").agg(
        recon_acc_mean=("recon_acc", "mean"),
        recon_acc_std=("recon_acc", "std"),
        recon_CE_mean=("recon_CE", "mean"),
        recon_CE_std=("recon_CE", "std"),
        compression_pct_mean=("compression_pct", "mean"),
        compression_pct_std=("compression_pct", "std"),
    )

    results[name] = {
        "best_score_row": best_score_row,
        "best_acc_row": best_acc_row,
        "best_ce_row": best_ce_row,
        "best_compression_row": best_compression_row,
        "stats": stats
    }

print("=== Task-Awareness and Performance Summary (Best Seed + Mean ± Std) ===\n")
for name, r in results.items():
    print(f"{name}:")

    def fmt_val_std(val, std):
        return f"{val:.4f} ± {std:.4f}" if not np.isnan(std) else f"{val:.4f}"

    for metric_name, row in [
        ("Best Task-Aware", r["best_score_row"]),
        ("Best Accuracy", r["best_acc_row"]),
        ("Best CE", r["best_ce_row"]),
        ("Best Compression", r["best_compression_row"]),
    ]:
        alpha = row["alpha"]
        seed = row["seed"]
        recon_acc_mean, recon_acc_std = r["stats"].loc[alpha, ["recon_acc_mean", "recon_acc_std"]]
        recon_CE_mean, recon_CE_std = r["stats"].loc[alpha, ["recon_CE_mean", "recon_CE_std"]]
        compression_mean, compression_std = r["stats"].loc[alpha, ["compression_pct_mean", "compression_pct_std"]]

        print(f"  {metric_name} α: {alpha:.3f}, Seed: {seed}")
        if metric_name != "Best Compression":
            print(f"    Accuracy: {fmt_val_std(row['recon_acc'], recon_acc_std)}")
            print(f"    CE: {fmt_val_std(row['recon_CE'], recon_CE_std)}")
        else:
            print(f"    Compression: {fmt_val_std(row['compression_pct'], compression_std)}")
            print(f"    Accuracy: {fmt_val_std(row['recon_acc'], recon_acc_std)}")
            print(f"    CE: {fmt_val_std(row['recon_CE'], recon_CE_std)}")

        print(f"    Recon Size: {row.get('recon_avg_KB', np.nan):.2f} KB")
        print(f"    Size-Matched Accuracy: {row.get('size_matched_lossy_acc', np.nan):.4f}")
        print(f"    Size-Matched CE: {row.get('size_matched_lossy_CE', np.nan):.4f}")
        print()
    print("-" * 60)


=== Task-Awareness and Performance Summary (Best Seed + Mean ± Std) ===

Simple Tanh:
  Best Task-Aware α: 0.410, Seed: 43.0
    Accuracy: 0.7840 ± 0.0026
    CE: 1.2725 ± 0.0408
    Recon Size: 1.32 KB
    Size-Matched Accuracy: 0.4588
    Size-Matched CE: 2.4264

  Best Accuracy α: 0.410, Seed: 43.0
    Accuracy: 0.7840 ± 0.0026
    CE: 1.2725 ± 0.0408
    Recon Size: 1.32 KB
    Size-Matched Accuracy: 0.4588
    Size-Matched CE: 2.4264

  Best CE α: 0.410, Seed: 43.0
    Accuracy: 0.7840 ± 0.0026
    CE: 1.2725 ± 0.0408
    Recon Size: 1.32 KB
    Size-Matched Accuracy: 0.4588
    Size-Matched CE: 2.4264

  Best Compression α: 0.410, Seed: 43.0
    Compression: 9.3064 ± 1.5707
    Accuracy: 0.7840 ± 0.0026
    CE: 1.2725 ± 0.0408
    Recon Size: 1.32 KB
    Size-Matched Accuracy: 0.4588
    Size-Matched CE: 2.4264

------------------------------------------------------------
Simple Sigmoid:
  Best Task-Aware α: 0.350, Seed: 43.0
    Accuracy: 0.7835 ± 0.0013
    CE: 1.2697 ± 0.0404


In [4]:
import pandas as pd
import numpy as np

dfs = {
    "Task-Aware": [],
    "Accuracy": [],
    "CE": [],
    "Compression": []
}

for name, r in results.items():
    def fmt_val_std(val, std):
        return f"{val:.4f} ± {std:.4f}" if not np.isnan(std) else f"{val:.4f}"

    for metric_name, row in [
        ("Task-Aware", r["best_score_row"]),
        ("Accuracy", r["best_acc_row"]),
        ("CE", r["best_ce_row"]),
        ("Compression", r["best_compression_row"]),
    ]:
        alpha = row["alpha"]
        seed = row["seed"]
        recon_acc_mean, recon_acc_std = r["stats"].loc[alpha, ["recon_acc_mean", "recon_acc_std"]]
        recon_CE_mean, recon_CE_std = r["stats"].loc[alpha, ["recon_CE_mean", "recon_CE_std"]]
        compression_mean, compression_std = r["stats"].loc[alpha, ["compression_pct_mean", "compression_pct_std"]]

        dfs[metric_name].append({
            "Decoder": name,
            "Alpha": alpha,
            "Seed": seed,
            "Accuracy (Best ± Std)": fmt_val_std(row['recon_acc'], recon_acc_std),
            "CE (Best ± Std)": fmt_val_std(row['recon_CE'], recon_CE_std),
            "Compression (%) (Best ± Std)": fmt_val_std(row['compression_pct'], compression_std),
            "Recon Size (KB)": row.get('recon_avg_KB', np.nan),
            "Size-Matched Accuracy": row.get('size_matched_lossy_acc', np.nan),
            "Size-Matched CE": row.get('size_matched_lossy_CE', np.nan)
        })

# Convert to DataFrames
task_aware_df = pd.DataFrame(dfs["Task-Aware"])
accuracy_df = pd.DataFrame(dfs["Accuracy"])
ce_df = pd.DataFrame(dfs["CE"])
compression_df = pd.DataFrame(dfs["Compression"])

# Display
print("=== Task-Aware Best Variants ===")
display(task_aware_df)
print("\n=== Accuracy Best Variants ===")
display(accuracy_df)
print("\n=== CE Best Variants ===")
display(ce_df)
print("\n=== Compression Best Variants ===")
display(compression_df)

# Optionally save to CSV
task_aware_df.to_csv("best_task_aware.csv", index=False)
accuracy_df.to_csv("best_accuracy.csv", index=False)
ce_df.to_csv("best_ce.csv", index=False)
compression_df.to_csv("best_compression.csv", index=False)


=== Task-Aware Best Variants ===


,Decoder,Alpha,Seed,Accuracy (Best ± Std),CE (Best ± Std),Compression (%) (Best ± Std),Recon Size (KB),Size-Matched Accuracy,Size-Matched CE
0,Simple Tanh,0.41,43.0,0.7840 ± 0.0026,1.2725 ± 0.0408,9.3064 ± 1.5707,1.315476,0.4588,2.426366
1,Simple Sigmoid,0.35,43.0,0.7835 ± 0.0013,1.2697 ± 0.0404,15.8382 ± 3.6957,1.220735,0.4558,2.440055
2,MR Tanh,0.18,43.0,0.7818 ± 0.0057,1.4322 ± 0.0321,0.7670 ± 3.1838,1.439336,0.4588,2.413288
3,MR Sigmoid,0.08,409.0,0.7813 ± 0.0041,1.3515 ± 0.0586,0.6571 ± 2.8464,1.440930,0.4591,2.413527



=== Accuracy Best Variants ===


,Decoder,Alpha,Seed,Accuracy (Best ± Std),CE (Best ± Std),Compression (%) (Best ± Std),Recon Size (KB),Size-Matched Accuracy,Size-Matched CE
0,Simple Tanh,0.41,43.0,0.7840 ± 0.0026,1.2725 ± 0.0408,9.3064 ± 1.5707,1.315476,0.4588,2.426366
1,Simple Sigmoid,0.35,43.0,0.7835 ± 0.0013,1.2697 ± 0.0404,15.8382 ± 3.6957,1.220735,0.4558,2.440055
2,MR Tanh,0.18,43.0,0.7818 ± 0.0057,1.4322 ± 0.0321,0.7670 ± 3.1838,1.439336,0.4588,2.413288
3,MR Sigmoid,0.08,409.0,0.7813 ± 0.0041,1.3515 ± 0.0586,0.6571 ± 2.8464,1.440930,0.4591,2.413527



=== CE Best Variants ===


,Decoder,Alpha,Seed,Accuracy (Best ± Std),CE (Best ± Std),Compression (%) (Best ± Std),Recon Size (KB),Size-Matched Accuracy,Size-Matched CE
0,Simple Tanh,0.41,43.0,0.7840 ± 0.0026,1.2725 ± 0.0408,9.3064 ± 1.5707,1.315476,0.4588,2.426366
1,Simple Sigmoid,0.35,43.0,0.7835 ± 0.0013,1.2697 ± 0.0404,15.8382 ± 3.6957,1.220735,0.4558,2.440055
2,MR Tanh,0.18,43.0,0.7818 ± 0.0057,1.4322 ± 0.0321,0.7670 ± 3.1838,1.439336,0.4588,2.413288
3,MR Sigmoid,0.08,409.0,0.7813 ± 0.0041,1.3515 ± 0.0586,0.6571 ± 2.8464,1.440930,0.4591,2.413527



=== Compression Best Variants ===


,Decoder,Alpha,Seed,Accuracy (Best ± Std),CE (Best ± Std),Compression (%) (Best ± Std),Recon Size (KB),Size-Matched Accuracy,Size-Matched CE
0,Simple Tanh,0.41,43.0,0.7840 ± 0.0026,1.2725 ± 0.0408,9.3064 ± 1.5707,1.315476,0.4588,2.426366
1,Simple Sigmoid,0.35,43.0,0.7835 ± 0.0013,1.2697 ± 0.0404,15.8382 ± 3.6957,1.220735,0.4558,2.440055
2,MR Tanh,0.27,2.0,0.7772 ± 0.0020,1.4436 ± 0.0811,0.8318 ± 2.0344,1.438396,0.4589,2.413526
3,MR Sigmoid,0.32,409.0,0.7805 ± 0.0059,1.4129 ± 0.0574,7.9532 ± 2.6205,1.335103,0.4584,2.424368


In [5]:
results = []

for name, data in decoder_data.items():
    df = data["eval"].copy()
    df["task_awareness_score"] = df["recon_acc"] / (df["recon_CE"] + 1e-6)

    best_score_idx = df["task_awareness_score"].idxmax()
    best_acc_idx = df["recon_acc"].idxmax()
    best_ce_idx = df["recon_CE"].idxmin()
    best_compression_idx = df["compression_pct"].idxmax()

    bests = {
        "Task-Aware": df.loc[best_score_idx],
        "Accuracy": df.loc[best_acc_idx],
        "CE": df.loc[best_ce_idx],
        "Compression": df.loc[best_compression_idx],
    }

    for category, row in bests.items():
        # Decide which metrics to use based on compression
        if row.get("compression_pct", 0) >= 0:
            acc = row.get("size_matched_lossy_acc", row.get("baseline_acc", np.nan))
            ce = row.get("size_matched_lossy_CE", row.get("baseline_CE", np.nan))
        else:
            acc = row.get("baseline_acc", np.nan)
            ce = row.get("baseline_CE", np.nan)

        results.append({
            "Decoder": name,
            "Category": category,
            "Alpha": row["alpha"],

            # task-awareness
            "Task_Aware_Score": row["recon_acc"] / (row["recon_CE"] + 1e-6),

            # baseline metrics
            "Baseline_Accuracy": row.get("baseline_acc", np.nan),
            "Baseline_CE": row.get("baseline_CE", np.nan),

            # reconstruction metrics
            "Recon_Accuracy": row.get("recon_acc", np.nan),
            "Recon_CE": row.get("recon_CE", np.nan),
            "Recon_Size_KB": row.get("recon_avg_KB", np.nan),
            "Compression_%": row.get("compression_pct", np.nan),

            # size-matched or fallback
            "Size_Matched_Accuracy": acc,
            "Size_Matched_CE": ce,

            # raw training metrics
            "Raw_CE": data["train"]["raw_CE"].mean() if "raw_CE" in data["train"].columns else np.nan,
            "Raw_MSE": data["train"]["raw_MSE"].mean() if "raw_MSE" in data["train"].columns else np.nan,
            "Norm_CE": data["train"]["norm_CE"].mean() if "norm_CE" in data["train"].columns else np.nan,
            "Norm_MSE": data["train"]["norm_MSE"].mean() if "norm_MSE" in data["train"].columns else np.nan,
            "Total_Loss": data["train"]["L_Total"].mean() if "L_Total" in data["train"].columns else np.nan
        })

# Make DataFrame
summary_df = pd.DataFrame(results)

# Display
print("=== Task-Awareness and Performance Summary ===\n")
display(summary_df)

# Save to CSV
summary_csv_path = "decoder_summary_results_full.csv"
summary_df.to_csv(summary_csv_path, index=False)
print(f"\nResults saved to: {summary_csv_path}")


=== Task-Awareness and Performance Summary ===



,Decoder,Category,Alpha,Task_Aware_Score,Baseline_Accuracy,Baseline_CE,Recon_Accuracy,Recon_CE,Recon_Size_KB,Compression_%,Size_Matched_Accuracy,Size_Matched_CE,Raw_CE,Raw_MSE,Norm_CE,Norm_MSE,Total_Loss
0,Simple Tanh,Task-Aware,0.01,0.676786,0.7771,0.824283,0.7705,1.138468,1.291829,10.936681,0.4582,2.428468,0.002463,0.987814,0.039126,0.986533,0.322522
1,Simple Tanh,Accuracy,0.41,0.616127,0.7771,0.824283,0.7840,1.272464,1.315476,9.306368,0.4588,2.426366,0.002463,0.987814,0.039126,0.986533,0.322522
2,Simple Tanh,CE,0.01,0.676786,0.7771,0.824283,0.7705,1.138468,1.291829,10.936681,0.4582,2.428468,0.002463,0.987814,0.039126,0.986533,0.322522
3,Simple Tanh,Compression,0.41,0.587088,0.7771,0.824283,0.7839,1.335234,1.254342,13.521164,0.4571,2.433723,0.002463,0.987814,0.039126,0.986533,0.322522
4,Simple Sigmoid,Task-Aware,0.97,0.655753,0.7771,0.824283,0.7759,1.183220,1.296270,10.630509,0.4591,2.427868,0.003761,0.944018,0.056991,0.975923,0.542462
5,Simple Sigmoid,Accuracy,0.35,0.617093,0.7771,0.824283,0.7835,1.269663,1.220735,15.838173,0.4558,2.440055,0.003761,0.944018,0.056991,0.975923,0.542462
6,Simple Sigmoid,CE,0.97,0.655753,0.7771,0.824283,0.7759,1.183220,1.296270,10.630509,0.4591,2.427868,0.003761,0.944018,0.056991,0.975923,0.542462
7,Simple Sigmoid,Compression,0.35,0.617093,0.7771,0.824283,0.7835,1.269663,1.220735,15.838173,0.4558,2.440055,0.003761,0.944018,0.056991,0.975923,0.542462
8,MR Tanh,Task-Aware,0.27,0.590580,0.7771,0.824283,0.7749,1.312098,1.475290,-1.711794,0.7771,0.824283,0.055118,0.955474,0.180895,0.936842,0.395847
9,MR Tanh,Accuracy,0.18,0.545868,0.7771,0.824283,0.7818,1.432214,1.439336,0.767018,0.4588,2.413288,0.055118,0.955474,0.180895,0.936842,0.395847



Results saved to: decoder_summary_results_full.csv


---